<a href="https://colab.research.google.com/github/KrathK9722/Analise-de-Dados-Python-Projeto-Avaliativo-M1W07/blob/main/Analise_Google_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1 - Importação das bibliotecas:**

In [429]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

---
## 2 - **Carregamento dos dados**:

In [430]:
id_arquivo = "1iNVCXfu2iFVVHlffBz-fky7U28NhhNv9"

url = f"https://drive.google.com/uc?export=download&id={id_arquivo}"

dados_originais = pd.read_csv(url,sep=";", encoding="latin1")

Sepação feita por ";" porque o padrão estava como "," o que fazia com que o CSV fosse importado com somente uma coluna.

# **3 - Visualização dos dados brutos:**

In [431]:
dados_originais.head(30)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
5,01/02/2019,1000,534,M,4,1,C,187,HIGIENE,HASTES FLEXIVEIS,NaN,NaN,NaN,NaN
6,01/02/2019,1000,534,M,4,1,C,163,ALIMENTOS,MORTADELA,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
8,01/02/2019,1000,534,M,4,1,C,95,LIMPEZA,AMACIANTE,NaN,NaN,NaN,NaN
9,01/02/2019,1000,534,M,4,1,C,198,BEBIDAS,ENERGETICO,NaN,NaN,NaN,NaN


# **3.1 - Quantide de Colunas e linhas**

In [432]:
linhas, colunas = dados_originais.shape

print(f"O número de colunas da tabela é {colunas} e o número total de linhas/registros é {linhas}.")

O número de colunas da tabela é 14 e o número total de linhas/registros é 830000.


-----

### O QUE É CADA COLUNA?

1. DATA: Data da compra;
2. CO_ID: Identificação do número de compra (número da nota fiscal);
3. CL_ID: Identificação do cliente (número do cliente);
4. CL_GENERO: Sexo biológico informado pelo cliente;
5. CL_EC: Estado civil do cliente:
    
    1: Casado ou união estával;
    
    2: Divorciado;
    
    3: Separado;
    
    4: Solteiro;
    
    5: Viúvo.
6. CL_FHL: Número de filhos do cliente;
7. CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
8. PR_ID: Código do produto (SKU) adquirido;
9. PR_CAT: Categoria do produto adquirido;
10. PR_NOME: Nome do produto adquirido.


Demais colunas não contem dados e devem ser removidas no processo de limpeza.

***`Informações retiradas do documento de analise exploratoria da base de varejo.csv`***

-----

# **3.2 - Tipos de dados e outras informações:**

In [433]:
dados_originais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  object 
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  object 
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  object 
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  object 
 9   PR_NOME      830000 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), object(5)
memory usage: 88.7+ MB


#**4 - Limpeza e Validação dos dados**

Copia do banco de dados original para garantir que os dados originais não sejam modificados e possam ser acessados assim como vieram, permitindo que a tabela seja modificada na nova copia.

In [434]:
dados_modificados = pd.read_csv(url,sep=";", encoding="latin1", na_values=["#N/D"])

Modificação na nova tabela para que o pandas entenda que #N/D é nulo e some ao procurar pelos valores nulos.

### **4.1 - Linhas Duplicadas**

In [435]:
duplicados = dados_modificados.duplicated()
print(f"Linhas duplicadas: {duplicados.sum()}")

Linhas duplicadas: 96553


Agora que observados que existem registros duplicados precisamos entender se eles realmente são registros duplicados ou apenas registros de um mesmo produto comprado 2 vezes pela mesma pessoa.

In [436]:
repeticoes_legitimas = dados_modificados.groupby(["CO_ID", "PR_ID"]).size()
print(repeticoes_legitimas[repeticoes_legitimas > 1].head(10))

CO_ID  PR_ID
1000   4        2
       11       2
       13       2
       69       2
       218      2
       225      2
1078   21       2
       23       2
       25       2
       36       2
dtype: int64


Aqui visualizamos registros de notas fiscais com o mesmo produto agora precisamos ver se o resto dos dados nesse registro são identicos.

In [437]:
exemplo = dados_modificados[(dados_modificados["CO_ID"] == 1000) & (dados_modificados["PR_ID"] == 4)]
print(exemplo)

          DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID  \
3   01/02/2019   1000    534         M      4       1      C      4   
40  01/02/2019   1000    534         M      4       1      C      4   

       PR_CAT  PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  
3   ALIMENTOS  ABACAXI          NaN          NaN          NaN          NaN  
40  ALIMENTOS  ABACAXI          NaN          NaN          NaN          NaN  


Diversos dados duplicados foram encontrados isso nos trás algumas questões, podemos remover os dados supondo que são erros de registro ou levando em consideração que a tabela não tem a coluna quantidade podemos entender que as linhas duplicadas de um mesmo registro da compra de produto esta relacionado a quantidade de produtos compradas sendo cada linha um poroduto. Como solução resolvi levar em consideração a segunda opção e criar uma coluna de quantidades removendo as linhas duplicadas para facilitar a análise.

Obs: Vi depois no desafio que a segunda opção era realmente a correta a se seguir.

In [438]:
dados_com_quantidade = dados_modificados.groupby(
    ["CO_ID", "CL_ID","CL_GENERO","CL_EC","CL_FHL","CL_SEG", "PR_ID", "PR_CAT", "PR_NOME","DATA"],
    as_index=False,
    dropna=False
).size()

dados_com_quantidade = dados_com_quantidade.rename(columns={
    "DATA":"DATA_BR"
})

# Renomeia a coluna criada pelo .size() pra um nome mais claro
dados_com_quantidade = dados_com_quantidade.rename(columns={"size": "QUANTIDADE"})
coluna_data = dados_com_quantidade.pop("DATA_BR")
dados_com_quantidade["DATA_BR"] = coluna_data

Criação de uma nova tabela com coluna quantidades juntando os itens comprados mais de uma vez pela mesma pessoa em uma só compra na coluna "QUANTIDADE" a partir da função groupby que automaticamente ja junta linhas iguais, nome "DATA" alterado para "DATA_BR" para facilitar o entendimento futuro e separação de data internacional e data brasileira na conversão da data para tipo DATE_TIME.
Também mudei a posição da coluna "DATA_BR" para posição de ultima coluna por preferencia pessoal de organização.

Observação: O comando GroupBy no pandas remove automaticamente as linhas com valores nulos então para garantir que isso não ocorresse sem as devidas verificações decidi deixar o atributo dropna como false para que o pandas não fizesse isso automaticamente, assim como removi as colunas sem nome ao criar a coluna quantidade ja que colunas sem nome são inuteis para nossa análise ainda mais sabendo que todos os seus valores eram vázios.

In [439]:
dados_com_quantidade = dados_com_quantidade.drop_duplicates()

In [440]:
duplicados_limpos = dados_com_quantidade.duplicated()
print(f"Linhas duplicadas: {duplicados_limpos.sum()}")

Linhas duplicadas: 0


In [441]:
dados_com_quantidade

,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,QUANTIDADE,DATA_BR
0,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,2,01/02/2019
1,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,2,01/02/2019
2,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,2,01/02/2019
3,1000,534,M,4,1,C,23,ALIMENTOS,COGUMELOS,1,01/02/2019
4,1000,534,M,4,1,C,24,ALIMENTOS,COPA SUINA,1,01/02/2019
...,...,...,...,...,...,...,...,...,...,...,...
733442,919822,155,F,2,0,B,212,LIMPEZA,CERA,2,19/08/2022
733443,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA,1,19/08/2022
733444,919822,155,F,2,0,B,225,ALIMENTOS,ATUM,1,19/08/2022
733445,919822,155,F,2,0,B,227,ALIMENTOS,ARROZ,2,19/08/2022


Feita a limpeza agrupamento e remoção das linhas duplicadas agora seguimos para
a validação de valores nulos.

### **4.2 - Valores Nulos**

In [442]:
nulos = dados_com_quantidade.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
CO_ID            0
CL_ID            0
CL_GENERO        0
CL_EC            0
CL_FHL           0
CL_SEG           0
PR_ID            0
PR_CAT        3228
PR_NOME       3228
QUANTIDADE       0
DATA_BR          0
dtype: int64


Podemos ver que existem diversas categorias e nome de produto nulos, sabendo disso precisamos descobrir se conseguimos preencher esses valores corretamente ou se é necessários remover eles.

In [443]:
mapa_categoria = dados_com_quantidade.dropna(subset=["PR_CAT"]).drop_duplicates("PR_ID").set_index("PR_ID")["PR_CAT"]

dados_com_quantidade["PR_CAT"] = dados_com_quantidade["PR_CAT"].fillna(dados_com_quantidade["PR_ID"].map(mapa_categoria))

Criação de um dicionário para tentar preencher as categorias e nomes nulos com o valor correto de acordo com o ID do produto procurnado produtos de mesmo ID na tabela para coletar os nomes e categorias referentes ao ID.

In [444]:
# Pega quantos produtos unicos tem categoria única
ids_com_nulo = dados_com_quantidade.loc[dados_com_quantidade["PR_CAT"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com categoria nula: {len(ids_com_nulo)}")

# Procura quantos desses produtos tem essa categoria preenchida em outra linha
ids_recuperaveis = dados_com_quantidade.loc[dados_com_quantidade["PR_ID"].isin(ids_com_nulo) & dados_com_quantidade["PR_CAT"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm categoria em outra linha: {len(ids_recuperaveis)}")

Quantidade de produtos únicos com categoria nula: 1
Desses, quantos têm categoria em outra linha: 0


In [445]:
# Pega quantos produtos únicos têm nome nulo
ids_com_nulo_nome = dados_com_quantidade.loc[dados_com_quantidade["PR_NOME"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com nome nulo: {len(ids_com_nulo_nome)}")

# Desses produtos, quantos têm o nome preenchido em outra linha
ids_recuperaveis_nome = dados_com_quantidade.loc[dados_com_quantidade["PR_ID"].isin(ids_com_nulo_nome) & dados_com_quantidade["PR_NOME"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm nome em outra linha: {len(ids_recuperaveis_nome)}")

Quantidade de produtos únicos com nome nulo: 1
Desses, quantos têm nome em outra linha: 0


Ao fazer essa visualização dos IDS unicos que tem categoria e nomes nulos podemos entender que todas essas categorias nulas vem de um mesmo produto então precisamos descobrir qual é esse produto.

In [446]:
pr_id_desconhecido = dados_com_quantidade.loc[dados_com_quantidade["PR_CAT"].isna(), "PR_ID"].unique()[0]
print(f"PR_ID: {pr_id_desconhecido}")

# Confirma que o nome do produto também está nulo
print(dados_com_quantidade.loc[dados_com_quantidade["PR_ID"] == pr_id_desconhecido, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

PR_ID: 107
     PR_ID PR_CAT PR_NOME
93     107    NaN     NaN
190    107    NaN     NaN
548    107    NaN     NaN
730    107    NaN     NaN
791    107    NaN     NaN


Agora que descobrimos o produto entendemos que esse produto em especifíco esta com algum problema em seus registros que faz com que o seu nome e categoria não estejam informados. Por isso para facilitar visualizações futuras vamos trocar os dados nesses espaços por "Sem Categoria" e por "Sem Nome" para que qualquer um consiga entender ao visualizar os dados.

In [447]:
dados_limpos = dados_com_quantidade.copy()
linhas, colunas = dados_limpos.shape
categorias_corrigidas = False
nomes_corrigidos = False
n_linha=0
for i in range(linhas):
  if pd.isna(dados_limpos.loc[i, "PR_CAT"]) and categorias_corrigidas == False:
    dados_limpos["PR_CAT"] = dados_limpos["PR_CAT"].fillna("Sem Categoria")
    categorias_corrigidas = True
  if pd.isna(dados_limpos.loc[i, "PR_NOME"]) and nomes_corrigidos == False:
    dados_limpos["PR_NOME"] = dados_limpos["PR_NOME"].fillna("Sem Nome")
    nomes_corrigidos = True
  if categorias_corrigidas == True and nomes_corrigidos == True:
    print("Correção concluida")
  else:
    n_linha += 1
    continue
  print(f"Número de linhas válidadas até a conclusão: {n_linha}")
  break

Correção concluida
Número de linhas válidadas até a conclusão: 93


Validação feita para procurar se algum valor em categoria ou em nome é nulo mesmo e após isso trocar todos os valores vazios por um aviso de "Sem Categoria" ou "Sem Nome" com um if final para verificar se as alterações ja foram feitas e cancelar o loop que levaria muito tempo para passar por todos os valores considerando o grande número de registros.

Obs: Essa etapa poderia ter sido feita utilizando apenas "dados_limpos["PR_CAT"] = dados_limpos["PR_CAT"].fillna("Sem Categoria")" e "dados_limpos["PR_NOME"] = dados_limpos["PR_NOME"].fillna("Sem Nome")" no entanto como o documento pede que seja feito o uso de If e Else achei interessante criar um loop que diga quantas linhas precisaram ser validadas para encontrar ao menos um nome nulo e uma categoria nula.

In [448]:
print(dados_limpos.loc[dados_com_quantidade["PR_ID"] == pr_id_desconhecido, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

     PR_ID         PR_CAT   PR_NOME
93     107  Sem Categoria  Sem Nome
190    107  Sem Categoria  Sem Nome
548    107  Sem Categoria  Sem Nome
730    107  Sem Categoria  Sem Nome
791    107  Sem Categoria  Sem Nome


In [449]:
nulos = dados_limpos.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
CO_ID         0
CL_ID         0
CL_GENERO     0
CL_EC         0
CL_FHL        0
CL_SEG        0
PR_ID         0
PR_CAT        0
PR_NOME       0
QUANTIDADE    0
DATA_BR       0
dtype: int64


Podemos ver que a alteração dos valores foi muito bem sucedida. Após a limpeza e validação de valores nulos e duplicados precisamos validas e converter as Datas de Registro.

### **4.3 - Conversão e Validação das Datas**

In [450]:
dados_data = dados_limpos.copy()

dados_data["DATA_INT"] = pd.to_datetime(dados_data["DATA_BR"], format="%d/%m/%Y", errors="coerce")

In [451]:
dados_data.head(3)

,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,QUANTIDADE,DATA_BR,DATA_INT
0,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,2,01/02/2019,2019-02-01
1,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,2,01/02/2019,2019-02-01
2,1000,534,M,4,1,C,13,ALIMENTOS,BANANA,2,01/02/2019,2019-02-01


In [452]:
datas_invalidas = dados_data["DATA_INT"].isna().sum()
print(f"Datas inválidas: {datas_invalidas}")

Datas inválidas: 0


Após a conversão e verificação das datas podemos seguir para a próxima etapa que é o agrupamento de informações relevantes e separação correta dos dados para análise.

# **5 - Agrupamento de Dados**

### **AGRUPAMENTO DE COMPRAS:**

In [459]:
compras_agrupadas = dados_data.groupby("CO_ID", as_index=False).agg(
    CL_ID=("CL_ID", "first"),           # O cliente é sempre o mesmo então podemos pegar somente o primeiro
    DATA_INT=("DATA_INT", "first"),       # A data da compra também é a mesma então podemos pegar o primeiro
    QTD_TOTAL_ITENS=("QUANTIDADE", "sum"),      # Soma quantidade total de itens comprados
    QTD_PRODUTOS_DIFERENTES=("PR_ID", "nunique"),# Soma quantidade de itens diferentes na compra
    PRODUTOS=("PR_NOME", list),         # Lista o nome de todos os produtos comprados
)

compras_agrupadas

,CO_ID,CL_ID,DATA_INT,QTD_TOTAL_ITENS,QTD_PRODUTOS_DIFERENTES,PRODUTOS
0,1000,534,2019-02-01,52,46,"[ABACAXI, AZEITE, BANANA, COGUMELOS, COPA SUIN..."
1,1040,279,2019-02-01,15,15,"[CAFE, LEITE CONDENSADO, MAMAO PAPAYA, OVOS, S..."
2,1078,290,2019-02-01,82,63,"[CHUPETA, ACHOCOLATADO, ALHO, BANANA, BATATA, ..."
3,1082,323,2019-02-01,71,62,"[ALMONDEGA, AZEITE, CAFE, DANETTE, FILE DE PEI..."
4,1103,957,2019-02-01,6,6,"[BATATA DOCE, DANETTE, DOCE, PROTETOR SOLAR, S..."
...,...,...,...,...,...,...
18466,919655,18,2022-08-19,83,67,"[ABACATE, ACHOCOLATADO, AZEITONA, BATATA, CAFE..."
18467,919682,332,2022-08-19,57,48,"[ALMONDEGA, ARROZ INTEGRAL, AZEITONA, BANANA, ..."
18468,919716,801,2022-08-19,81,65,"[ACHOCOLATADO, ARROZ, ARROZ INTEGRAL, ATUM, BA..."
18469,919770,888,2022-08-19,70,59,"[CHUPETA, MORDEDOR, ALHO, CAFE, CEBOLA, COPA S..."


Compreensão da RN relacionada a CO_ID fazendo agrupamento dos itens de uma mesma compra para uso dem análises posteriores.

### **AGRUPAMENTO DE COMPRAS:**